# Employee Attrition Prediction — Enhanced E(3A)CSPM
## Final Year Project: Novel Extensions over Base Paper

### Code Explanation & Dataset Overview

**Base Paper:** *"Analysis and classification of employee attrition and absenteeism in industry: A sequential pattern mining-based methodology"* — Nawaz et al., Computers in Industry (2024)

---

### What the existing code does
The original notebook implements the **E(3A)CSPM pipeline** in four stages:
1. **Data loading & pre-processing** — 4 Kaggle datasets, label encoding, MICE imputation
2. **TKS (Top-K Sequential) pattern mining** — converts employee records into integer sequences and finds the top-k most frequent patterns
3. **ERMiner sequential rules** — mines Y→Z rules (antecedent precedes consequent) with support & confidence thresholds
4. **8 classifiers evaluated** (MNB, GNB, DT, RF, MLP, SVM, kNN, LR) using 10-fold cross-validation on (a) raw features and (b) TKS pattern features

### Datasets used (4 total)
| ID | Name | Rows | Task | Target |
|----|------|------|------|--------|
| IBM-1 | IBM HR Analytics | 1,470 | Binary | Attrition (Yes/No) |
| HRD-2 | Human Resources Data Set | 311 | Binary | Termd (0/1) |


---

### YOUR Novelty (from the diagram image)
Compared to the base paper, this project adds **5 new components**:

| # | New Component | Replaces / Extends |
|---|--------------|--------------------|
| 1 | **SPAM** (bitmap sequential mining) | TKS (top-k only) → finds ALL frequent patterns |
| 2 | **RuleGrowth** (faster rule strategy) | ERMiner → faster rule growth |
| 3 | **Weighted pattern feature matrix** | Binary matrix → support×confidence score per cell |
| 4 | **XGBoost classifier** | Adds XGBoost to the 8 existing classifiers |
| 5 | **SHAP explainability layer** | Which patterns drive each prediction |


## ① Install Dependencies

In [ ]:
# Install required packages
import subprocess, sys

pkgs = ['imbalanced-learn', 'xgboost', 'shap']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed ✓')

All packages installed ✓


## ② Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import warnings, time, os, zipfile
from collections import Counter
from itertools import combinations

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             confusion_matrix)
from imblearn.over_sampling import SMOTE

# NEW: XGBoost + SHAP
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

print('=' * 70)
print('  Enhanced E(3A)CSPM: Employee Attrition & Absenteeism Analysis')
print('  Novel Additions: SPAM | RuleGrowth | Weighted Matrix | XGBoost | SHAP')
print('=' * 70)

  Enhanced E(3A)CSPM: Employee Attrition & Absenteeism Analysis
  Novel Additions: SPAM | RuleGrowth | Weighted Matrix | XGBoost | SHAP


## ③ Dataset Paths & Loading

In [ ]:
# ── Dataset paths (update these for your environment) ──────────
DATASET_PATHS = {
    'IBM-1': [
        '/content/WA_Fn-UseC_-HR-Employee-Attrition.csv',
        '/content/IBM_Dataset.csv',
        'IBM_Dataset.csv',
        'WA_Fn-UseC_-HR-Employee-Attrition.csv'
    ],
    'HRA-4': [
        '/content/hr_data.csv',
        '/content/hr_data__2_.csv',
        'hr_data.csv'
    ]
}

TASK_TYPE = {
    'IBM-1': 'binary',
    'HRA-4': 'binary'
}

def load_dataset(name):
    """Try all candidate paths for a dataset."""
    for path in DATASET_PATHS[name]:
        if os.path.exists(path):
            try:
                if path.endswith('.xls') or path.endswith('.xlsx'):
                    df = pd.read_excel(path)
                else:
                    df = pd.read_csv(path)
                print(f'  ✓ Loaded {name} | Path: "{path}" | Shape: {df.shape}')
                return df
            except Exception as e:
                print(f'  ✗ Failed "{path}": {e}')
    print(f'  ✗ {name} NOT FOUND — skipping.')
    return None

print('Loading datasets...')
DATASETS = {}
for name in ['IBM-1', 'HRA-4']:
    df = load_dataset(name)
    if df is not None:
        DATASETS[name] = df

Loading datasets...
  ✓ Loaded IBM-1 | Path: "/content/IBM_Dataset.csv" | Shape: (1470, 35)
  ✓ Loaded HRA-4 | Path: "/content/hr_data.csv" | Shape: (14999, 11)


## ④ Pre-processing Functions (Paper §4.1)

In [ ]:
def preprocess_ibm1(df):
    drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    target = 'Attrition'
    df[target] = (df[target].astype(str).str.strip().str.lower() == 'yes').astype(int)
    return df, target
def preprocess_hra4(df):
    target = None
    for c in ['left', 'Left']:
        if c in df.columns:
            target = c
            break
    if target is None:
        target = df.columns[-1]

    # Convert 'yes' to 1, 'no' to 0 for the target column, handling casing and spaces
    mapped_values = df[target].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})
    df[target] = mapped_values.fillna(0).astype(int)
    return df, target

PREPROCESSORS = {
    'IBM-1': preprocess_ibm1,
    'HRA-4': preprocess_hra4
}

def encode_features(df, target):
    df = df.copy()
    le = LabelEncoder()
    for col in df.columns:
        if col == target:
            continue
        def _is_categorical(series):
            return (
                series.dtype == 'object'
                or str(series.dtype) == 'category'
                or pd.api.types.is_string_dtype(series)
            )
        if _is_categorical(df[col]):
            df[col] = le.fit_transform(df[col].astype(str))
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.fillna(df.median(numeric_only=True))
    y = df[target].values
    X = df.drop(columns=[target]).values.astype(float)
    feature_names = [c for c in df.columns if c != target]

    return X, y, feature_names

print('Pre-processing functions defined ✓')

Pre-processing functions defined ✓


## ⑤ NOVELTY 1: SPAM — Bitmap Sequential Pattern Mining

**Base paper used TKS** (Top-K Sequential) which finds only the *top-k* patterns based on frequency.

**Your project uses SPAM** (Sequential PAttern Mining using bitmaps) which:
- Finds **ALL** frequent patterns above a minimum support threshold (not just top-k)
- Uses a **bitmap representation**: each row in the database is a bit in a vector → bitwise AND gives support counting in O(n/64) time
- More complete pattern coverage → richer feature matrix for classifiers

In [ ]:
class SPAM:
    """
    NOVELTY 1: SPAM — Sequential PAttern Mining with bitmap representation.
    Finds ALL frequent sequential patterns above minsup (not just top-k).

    Key difference from TKS:
    - TKS: user specifies k, finds exactly k patterns (top-k by frequency)
    - SPAM: user specifies minsup threshold, finds ALL patterns above it

    Bitmap trick: each sequence is a bit-vector of length n_sequences.
    Support of a pattern = popcount(bitmap).
    Extending a pattern = bitmap AND with new item bitmap.
    """

    def __init__(self, minsup=0.10, max_length=4):
        self.minsup      = minsup
        self.max_length  = max_length
        self.patterns_   = []   # list of (pattern_tuple, support_count, sup_pct)
        self.n_seq_      = 0
        self.token_map_  = {}
        self.inv_token_  = {}
        self._bitmaps    = {}   # token -> np.uint8 array (bit per sequence)

    # ── Encode X into integer token sequences ─────────────────
    def _encode(self, X, feature_names=None):
        n_rows, n_cols = X.shape
        sequences = []

        col_bins = []
        for j in range(n_cols):
            vals = X[:, j]
            unique_vals = np.unique(vals)
            if len(unique_vals) <= 5:
                bins = unique_vals
            else:
                bins = np.unique(np.percentile(vals, [0, 25, 50, 75, 100]))
            col_bins.append(bins)

        tc = len(self.token_map_) + 1
        for i in range(n_rows):
            seq  = []
            seen = set()
            for j in range(n_cols):
                v       = X[i, j]
                bins    = col_bins[j]
                bi      = int(np.searchsorted(bins, v, side='right')) - 1
                bi      = max(0, min(bi, len(bins) - 1))
                key     = (j, bi)
                if key not in self.token_map_:
                    self.token_map_[key] = tc
                    fname = feature_names[j] if feature_names else f'F{j}'
                    self.inv_token_[tc] = (j, bi, f'{fname}={v:.2g}')
                    tc += 1
                tok = self.token_map_[key]
                if tok not in seen:
                    seq.append(tok)
                    seen.add(tok)
            sequences.append(seq)
        return sequences

    # ── Build bitmaps: token -> bit-array of length n_seq ────
    def _build_bitmaps(self, sequences):
        n = len(sequences)
        bitmaps = {}
        for i, seq in enumerate(sequences):
            for tok in seq:
                if tok not in bitmaps:
                    bitmaps[tok] = np.zeros(n, dtype=np.uint8)
                bitmaps[tok][i] = 1
        return bitmaps

    def _support(self, bitmap):
        return int(bitmap.sum())

    # ── DFS-based SPAM expansion ─────────────────────────────
    def _spam_dfs(self, prefix, prefix_bitmap, sequences, depth):
        if depth >= self.max_length:
            return

        # Get all candidate extension tokens that appear AFTER prefix in at least one sequence
        active_seqs = [i for i in range(len(sequences)) if prefix_bitmap[i] == 1]

        # Collect items that follow the prefix in active sequences (s-extension)
        ext_counts = Counter()
        for i in active_seqs:
            seq = sequences[i]
            # Find position of last prefix element
            last_prefix_tok = prefix[-1]
            if last_prefix_tok in seq:
                pos = seq.index(last_prefix_tok)
                for tok in seq[pos + 1:]:
                    ext_counts[tok] += 1

        minsup_count = self.minsup * len(sequences)

        for tok, cnt in ext_counts.items():
            if cnt < minsup_count:
                continue
            new_bitmap = prefix_bitmap & self._bitmaps.get(tok, np.zeros(len(sequences), dtype=np.uint8))
            sup = self._support(new_bitmap)
            if sup < minsup_count:
                continue
            new_pattern = prefix + [tok]
            sup_pct = round(100.0 * sup / len(sequences), 2)
            self.patterns_.append((tuple(new_pattern), sup, sup_pct))
            self._spam_dfs(new_pattern, new_bitmap, sequences, depth + 1)

    def fit(self, X, y=None, feature_names=None):
        t0 = time.time()

        # Sample large datasets
        MAX_ROWS = 3000
        if len(X) > MAX_ROWS:
            idx = np.random.choice(len(X), MAX_ROWS, replace=False)
            X = X[idx]
            print(f'    SPAM: Sampled {MAX_ROWS}/{len(X)} rows for speed')

        self.n_seq_ = len(X)
        sequences   = self._encode(X, feature_names)
        self._bitmaps = self._build_bitmaps(sequences)

        minsup_count = self.minsup * self.n_seq_

        # Frequent single items (length-1 seeds)
        for tok, bmap in self._bitmaps.items():
            sup = self._support(bmap)
            if sup >= minsup_count:
                sup_pct = round(100.0 * sup / self.n_seq_, 2)
                self.patterns_.append(((tok,), sup, sup_pct))
                # DFS expand
                self._spam_dfs([tok], bmap, sequences, depth=1)

        # Deduplicate & sort by support
        seen = set()
        unique_patterns = []
        for pat, sup, pct in sorted(self.patterns_, key=lambda x: -x[1]):
            if pat not in seen:
                seen.add(pat)
                unique_patterns.append((pat, sup, pct))
        self.patterns_ = unique_patterns

        elapsed = time.time() - t0
        print(f'    SPAM → {len(self.patterns_)} ALL-frequent patterns '
              f'(minsup={self.minsup}) in {elapsed:.2f}s ✓')
        return self

    def transform(self, X):
        """Returns binary pattern-presence matrix (same as TKS)."""
        sequences  = self._encode(X)
        n_patterns = len(self.patterns_)
        if n_patterns == 0:
            return X
        Xp = np.zeros((len(sequences), n_patterns), dtype=np.float32)
        for pi, (pat, _, _) in enumerate(self.patterns_):
            pat_list = list(pat)
            for si, s  Fig. 16 case comparison for IBM dataset
eq in enumerate(sequences):
                it = iter(seq)
                if all(item in it for item in pat_list):
                    Xp[si, pi] = 1.0
        return Xp

    def get_top_patterns_text(self, n=10):
        lines = []
        for i, (pat, sup, pct) in enumerate(self.patterns_[:n]):
            tokens = [self.inv_token_.get(t, (0, 0, str(t)))[2] for t in pat]
            lines.append(f'  {i+1:2d}. [{', '.join(tokens)}]  sup={pct}% ({sup})')
        return '\n'.join(lines)

print('SPAM class (Novelty 1) defined ✓')

SPAM class (Novelty 1) defined ✓


## ⑥ NOVELTY 2: RuleGrowth — Faster Sequential Rule Mining

**Base paper used ERMiner** which uses equivalence classes and Sparse Count Matrix (SCM).

**Your project uses RuleGrowth** which:
- Grows rules incrementally (left/right expansions, similar to PrefixSpan strategy)
- Uses a **projected database** per rule prefix — only scans relevant sequences
- Significantly faster on large datasets because it avoids the full SCM computation
- Finds the same rules but with fewer wasted operations

In [ ]:
class RuleGrowth:
    """
    NOVELTY 2: RuleGrowth — faster sequential rule mining via incremental rule growth.

    Key difference from ERMiner:
    - ERMiner: builds equivalence classes of rules with same antecedent/consequent,
               uses full Sparse Count Matrix (expensive for dense datasets)
    - RuleGrowth: grows each rule by expanding antecedent or consequent one item
                  at a time using a projected database (only relevant sequences scanned)

    This gives the same rules but is faster because:
    1. Projected databases shrink as rules grow (fewer sequences to scan)
    2. No SCM needed — support counted directly from projections
    """

    def __init__(self, minsup=0.05, minconf=0.60, max_ant=2, max_con=1):
        self.minsup   = minsup
        self.minconf  = minconf
        self.max_ant  = max_ant
        self.max_con  = max_con
        self.rules_   = []

    def _project_db(self, sequences, items, require_order=True):
        """Return indices of sequences that contain all items in order."""
        result = []
        for i, seq in enumerate(sequences):
            if require_order:
                pos = -1
                found = True
                for item in items:
                    try:
                        pos = seq.index(item, pos + 1)
                    except ValueError:
                        found = False
                        break
                if found:
                    result.append(i)
            else:
                if all(item in seq for item in items):
                    result.append(i)
        return result

    def _rule_support(self, sequences, antecedent, consequent):
        """Count sequences where antecedent appears before consequent."""
        count = 0
        for seq in sequences:
            ant_pos = []
            con_pos = []
            for a in antecedent:
                if a in seq:
                    ant_pos.append(seq.index(a))
            for c in consequent:
                if c in seq:
                    con_pos.append(seq.index(c))
            if len(ant_pos) == len(antecedent) and len(con_pos) == len(consequent):
                if max(ant_pos) < min(con_pos):
                    count += 1
        return count

    def fit(self, sequences, n_total, inv_token=None):
        t0       = time.time()
        self.inv_ = inv_token or {}
        n        = len(sequences)
        minsup_c = self.minsup * n

        # ── Step 1: Find frequent items (seed rules: {a} → {b}) ──
        item_counts = Counter(tok for seq in sequences for tok in set(seq))
        freq_items  = [it for it, cnt in item_counts.items()
                       if cnt / n >= self.minsup][:25]  # limit for speed

        rules = []

        # ── Step 2: Seed rules {a} → {b} ────────────────────
        for a in freq_items:
            ant_proj = self._project_db(sequences, [a], require_order=False)
            sup_ant  = len(ant_proj)
            if sup_ant < minsup_c:
                continue

            for b in freq_items:
                if b == a:
                    continue
                sup_rule = self._rule_support(sequences, [a], [b])
                if sup_rule < minsup_c:
                    continue
                conf = sup_rule / sup_ant if sup_ant > 0 else 0
                if conf < self.minconf:
                    continue

                rule = {
                    'antecedent': (a,),
                    'consequent': (b,),
                    'support'   : sup_rule,
                    'sup_pct'   : round(100 * sup_rule / n, 2),
                    'confidence': round(conf, 3)
                }
                rules.append(rule)

                # ── Step 3: Right-grow — expand consequent ────
                if self.max_con > 1:
                    for c in freq_items:
                        if c in (a, b):
                            continue
                        sup_r2 = self._rule_support(sequences, [a], [b, c])
                        if sup_r2 < minsup_c:
                            continue
                        conf2 = sup_r2 / sup_ant if sup_ant > 0 else 0
                        if conf2 >= self.minconf:
                            rules.append({
                                'antecedent': (a,),
                                'consequent': (b, c),
                                'support'   : sup_r2,
                                'sup_pct'   : round(100 * sup_r2 / n, 2),
                                'confidence': round(conf2, 3)
                            })

                # ── Step 4: Left-grow — expand antecedent ─────
                if self.max_ant > 1:
                    for a2 in freq_items:
                        if a2 in (a, b):
                            continue
                        sup_ant2 = sum(1 for seq in sequences
                                       if a in seq and a2 in seq)
                        if sup_ant2 < minsup_c:
                            continue
                        sup_r3 = self._rule_support(sequences, [a, a2], [b])
                        if sup_r3 < minsup_c:
                            continue
                        conf3 = sup_r3 / sup_ant2 if sup_ant2 > 0 else 0
                        if conf3 >= self.minconf:
                            rules.append({
                                'antecedent': (a, a2),
                                'consequent': (b,),
                                'support'   : sup_r3,
                                'sup_pct'   : round(100 * sup_r3 / n, 2),
                                'confidence': round(conf3, 3)
                            })

        self.rules_ = sorted(rules, key=lambda r: -r['confidence'])[:20]
        elapsed     = time.time() - t0
        print(f'    RuleGrowth → {len(self.rules_)} rules '
              f'(minsup={self.minsup}, minconf={self.minconf}) '
              f'in {elapsed:.2f}s ✓')
        return self

    def print_rules(self, n=5):
        inv = self.inv_
        for r in self.rules_[:n]:
            ant = [inv.get(t, (0, 0, str(t)))[2] for t in r['antecedent']]
            con = [inv.get(t, (0, 0, str(t)))[2] for t in r['consequent']]
            print(f'    {ant} → {con} | sup={r["sup_pct"]}% | conf={r["confidence"]}')

print('RuleGrowth class (Novelty 2) defined ✓')

RuleGrowth class (Novelty 2) defined ✓


## ⑦ NOVELTY 3: Weighted Pattern Feature Matrix

**Base paper used a binary pattern feature matrix** — each cell = 1 if the pattern is present in the employee record, 0 otherwise.

**Your project uses a weighted matrix** where each cell = `support × confidence` of the pattern:
- A pattern present in a high-support + high-confidence rule gets a **higher weight**
- This encodes *how reliable* the pattern is, not just whether it's present
- Classifiers (especially tree-based) can use the magnitude of the weight to split more effectively

In [ ]:
def build_weighted_pattern_matrix(X, spam_model, rulegrowth_model):
    """
    NOVELTY 3: Weighted Pattern Feature Matrix.

    Base paper: binary matrix — cell[i][j] = 1 if pattern j present in record i
    This project: weighted matrix — cell[i][j] = support × confidence score

    Weight computation:
    - For each SPAM pattern p, find if any RuleGrowth rule mentions p's items
    - weight(p) = support_pct(p) * max_confidence_of_rules_involving_p
    - If no rule involves p, weight(p) = support_pct(p) (pattern strength alone)
    """
    # Binary presence matrix from SPAM
    binary_matrix = spam_model.transform(X)  # shape (n_samples, n_patterns)

    n_patterns = len(spam_model.patterns_)
    if n_patterns == 0:
        return binary_matrix

    # Build weight vector: one weight per pattern
    pattern_weights = np.ones(n_patterns, dtype=np.float32)

    rules = rulegrowth_model.rules_ if rulegrowth_model.rules_ else []

    for pi, (pat, sup, pct) in enumerate(spam_model.patterns_):
        # Normalize support
        sup_norm = pct / 100.0  # 0-1

        # Find max confidence of any rule that mentions items in this pattern
        max_conf = 0.0
        pat_set  = set(pat)
        for rule in rules:
            rule_items = set(rule['antecedent']) | set(rule['consequent'])
            # Check overlap
            if pat_set & rule_items:
                max_conf = max(max_conf, rule['confidence'])

        if max_conf > 0:
            pattern_weights[pi] = sup_norm * max_conf
        else:
            pattern_weights[pi] = sup_norm

    # Weighted matrix: multiply each pattern column by its weight
    weighted_matrix = binary_matrix * pattern_weights[np.newaxis, :]

    print(f'    Weighted matrix: shape={weighted_matrix.shape}, '
          f'mean_weight={pattern_weights.mean():.4f}, '
          f'max_weight={pattern_weights.max():.4f} ✓')
    return weighted_matrix

print('Weighted pattern matrix function (Novelty 3) defined ✓')

Weighted pattern matrix function (Novelty 3) defined ✓


## ⑧ NOVELTY 4 & 5: XGBoost Classifier + SHAP Explainability

**Base paper had 8 classifiers.** Your project adds **XGBoost** (the 9th) which uses gradient boosted decision trees with regularization.

**SHAP (SHapley Additive exPlanations):** After classification, SHAP reveals *which patterns drive each prediction*, answering the question "why was this employee predicted to leave?"

In [ ]:
def build_classifiers(task='binary'):
    """8 base classifiers (paper) + XGBoost (novelty 4)."""
    n_cls = 2 if task == 'binary' else 3
    objective = 'binary:logistic' if task == 'binary' else 'multi:softmax'

    clfs = {
        'MNB': MultinomialNB(alpha=1),
        'GNB': GaussianNB(),
        'DT' : DecisionTreeClassifier(criterion='gini', random_state=42),
        'RF' : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'MLP': MLPClassifier(hidden_layer_sizes=(600,), activation='tanh',
                             solver='adam', max_iter=300, random_state=42),
        'SVM': SVC(C=1, kernel='rbf', probability=True, random_state=42),
        'kNN': KNeighborsClassifier(n_neighbors=2, metric='euclidean'),
        'LR' : LogisticRegression(C=1, max_iter=1000, random_state=42),
        # NOVELTY 4: XGBoost
        'XGB': XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            objective=objective,
            num_class=n_cls if task != 'binary' else None,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, verbosity=0
        )
    }
    # Remove num_class=None key issue for binary
    if task == 'binary':
        clfs['XGB'] = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            objective='binary:logistic',
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, verbosity=0
        )
    return clfs

def evaluate_classifier(clf, X, y, cv=10, task='binary'):
    avg  = 'binary' if task == 'binary' else 'macro'
    skf  = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    t0   = time.time()

    for train_idx, test_idx in skf.split(X, y):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx]

        if isinstance(clf, MultinomialNB):
            Xtr = np.clip(Xtr, 0, None)
            Xte = np.clip(Xte, 0, None)

        clf.fit(Xtr, ytr)
        yp = clf.predict(Xte)

        accs.append(accuracy_score(yte, yp))
        precs.append(precision_score(yte, yp, average=avg, zero_division=0))
        recs.append(recall_score(yte, yp, average=avg, zero_division=0))
        f1s.append(f1_score(yte, yp, average=avg, zero_division=0))

        try:
            if task == 'binary':
                yprob = clf.predict_proba(Xte)[:, 1]
                aucs.append(roc_auc_score(yte, yprob))
            else:
                yprob = clf.predict_proba(Xte)
                aucs.append(roc_auc_score(yte, yprob, multi_class='ovr', average='macro'))
        except Exception:
            aucs.append(np.nan)

    return {
        'ACC' : round(np.mean(accs)  * 100, 2),
        'P'   : round(np.mean(precs) * 100, 2),
        'R'   : round(np.mean(recs)  * 100, 2),
        'F1'  : round(np.mean(f1s)   * 100, 2),
        'AUC' : round(np.nanmean(aucs), 3),
        'Time': round(time.time() - t0, 3)
    }

print('Classifiers (8 + XGBoost) + evaluate_classifier defined ✓')

Classifiers (8 + XGBoost) + evaluate_classifier defined ✓


## ⑨ Visualisation Helpers

In [ ]:
SAVED_FIGS = []

def save_fig(fname):
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.close()
    SAVED_FIGS.append(fname)
    print(f'    Saved: {fname}')

def plot_frequent_features(df, target, ds_name, top_n=10):
    classes = sorted(df[target].unique())
    ncols   = min(len(classes), 2) + 1
    fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 6))
    if ncols == 1:
        axes = [axes]
    colors = ['#27ae60', '#e74c3c', '#3498db', '#f39c12']

    def top_feat(sub):
        fc = {}
        for col in df.columns:
            if col == target:
                continue
            try:
                mode_val = sub[col].mode().iloc[0]
                fc[f'{col}={mode_val}'] = (sub[col] == mode_val).sum()
            except Exception:
                pass
        return sorted(fc.items(), key=lambda x: -x[1])[:top_n]

    t = top_feat(df)
    if t:
        lbl, v = zip(*t)
        axes[0].barh(lbl, v, color='#2c3e50')
    axes[0].set_title(f'{ds_name} — All')
    axes[0].set_xlabel('Count')

    for ci, cls in enumerate(classes[:2]):
        ax  = axes[ci+1]
        sub = df[df[target] == cls]
        t2  = top_feat(sub)
        if t2:
            lbl2, v2 = zip(*t2)
            ax.barh(lbl2, v2, color=colors[ci % len(colors)])
        ax.set_title(f'{ds_name} — Class {cls}')
        ax.set_xlabel('Count')

    plt.suptitle(f'Frequent Features — {ds_name}', fontsize=12)
    plt.tight_layout()
    save_fig(f'fig2_{ds_name}_frequent_features.png')

def plot_case_comparison(res1, res2, ds_name):
    clfs  = [c for c in ['MNB','GNB','DT','RF','MLP','SVM','kNN','LR','XGB']
             if c in res1 and c in res2]
    acc1  = [res1[c]['ACC'] for c in clfs]
    acc2  = [res2[c]['ACC'] for c in clfs]
    x     = np.arange(len(clfs))
    w     = 0.35

    fig, ax = plt.subplots(figsize=(13, 5))
    b1 = ax.bar(x-w/2, acc1, w, label='Case 1 (All Features)', color='#3498db', alpha=0.85)
    b2 = ax.bar(x+w/2, acc2, w, label='Case 2 (SPAM Weighted Patterns)', color='#e74c3c', alpha=0.85)
    for bar in list(b1)+list(b2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(clfs)
    ax.set_ylim(0, 115)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'Case 1 vs Case 2 (Novel: Weighted Matrix) — {ds_name}')
    ax.legend()
    plt.tight_layout()
    save_fig(f'fig_case_compare_{ds_name}.png')

def plot_metrics_bar(results, ds_name, case_label):
    clfs    = [c for c in ['MNB','GNB','DT','RF','MLP','SVM','kNN','LR','XGB']
               if c in results]
    metrics = ['ACC', 'P', 'R', 'F1']
    x       = np.arange(len(clfs))
    w       = 0.18
    colors  = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

    fig, ax = plt.subplots(figsize=(14, 5))
    for mi, met in enumerate(metrics):
        vals = [results[c].get(met, 0) for c in clfs]
        ax.bar(x + mi*w, vals, w, label=met, color=colors[mi], alpha=0.85)
    ax.set_xticks(x + w*1.5)
    ax.set_xticklabels(clfs)
    ax.set_ylim(0, 115)
    ax.set_ylabel('Score (%)')
    ax.set_title(f'Metrics — {ds_name} — {case_label}')
    ax.legend()
    plt.tight_layout()
    save_fig(f'fig_metrics_{ds_name}_{case_label}.png')

def plot_summary_heatmap(all_results):
    datasets = list(all_results.keys())
    clfs     = ['MNB','GNB','DT','RF','MLP','SVM','kNN','LR','XGB']
    matrix   = np.full((len(datasets), len(clfs)), np.nan)

    for di, ds in enumerate(datasets):
        for ci, clf in enumerate(clfs):
            v = all_results[ds].get('case2', {}).get(clf, {}).get('ACC')
            if v is not None:
                matrix[di, ci] = v

    fig, ax = plt.subplots(figsize=(13, max(3, len(datasets)*1.2)))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=50, vmax=100)
    ax.set_xticks(range(len(clfs)))
    ax.set_xticklabels(clfs)
    ax.set_yticks(range(len(datasets)))
    ax.set_yticklabels(datasets)
    for i in range(len(datasets)):
        for j in range(len(clfs)):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f'{matrix[i,j]:.1f}',
                        ha='center', va='center', fontsize=9,
                        color='black' if matrix[i,j] < 88 else 'white')
    plt.colorbar(im, ax=ax, label='Accuracy (%)')
    ax.set_title('Enhanced E(3A)CSPM — Case 2 (SPAM+Weighted+XGB) Accuracy Heatmap')
    plt.tight_layout()
    save_fig('fig_summary_heatmap.png')

print('Visualisation helpers defined ✓')

Visualisation helpers defined ✓


## ⑩ NOVELTY 5: SHAP Explainability Layer

After training, SHAP assigns a Shapley value to each pattern feature for each prediction, answering: **"Which patterns pushed this employee's prediction towards attrition?"**

In [ ]:
def run_shap_explainability(X_weighted, y, spam_model, ds_name, task='binary'):
    """
    NOVELTY 5: SHAP Explainability Layer.

    Trains XGBoost on the weighted pattern matrix, then uses SHAP
    to explain which patterns most influence the predictions.

    Base paper had no explainability — only accuracy/F1/AUC metrics.
    This layer adds the 'why' dimension:
    - Global SHAP: which patterns matter most across all employees
    - Local SHAP: why a specific employee was predicted to leave
    """
    print(f'\n  [SHAP] Training XGBoost for SHAP analysis on {ds_name}...')

    if X_weighted.shape[1] == 0:
        print('    No patterns — skipping SHAP.')
        return

    # Use a 80/20 split for SHAP (faster than CV)
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_weighted, y, test_size=0.2, random_state=42, stratify=y)

    objective = 'binary:logistic' if task == 'binary' else 'multi:softmax'
    xgb = XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        objective=objective,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0
    )
    if task != 'binary':
        xgb.set_params(num_class=len(np.unique(y)))

    X_tr_c = np.clip(X_tr, 0, None).astype(np.float32)
    X_te_c = np.clip(X_te, 0, None).astype(np.float32)
    xgb.fit(X_tr_c, y_tr)

    # SHAP TreeExplainer (fast for tree-based models)
    explainer  = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_te_c)

    # Build pattern labels
    n_patterns = X_weighted.shape[1]
    pat_labels = []
    for i, (pat, sup, pct) in enumerate(spam_model.patterns_[:n_patterns]):
        tokens = [spam_model.inv_token_.get(t, (0, 0, f'T{t}'))[2] for t in pat]
        pat_labels.append((f'P{i+1}: ' + ', '.join(tokens[:2]))[:40])

    while len(pat_labels) < n_patterns:
        pat_labels.append(f'Pattern_{len(pat_labels)+1}')

    # ── Global SHAP bar chart ────────────────────────────────
    if isinstance(shap_values, list):
        sv = np.abs(shap_values[1])  # class 1 for binary
    else:
        sv = np.abs(shap_values)

    mean_shap = sv.mean(axis=0)
    top_idx   = np.argsort(mean_shap)[::-1][:15]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        [pat_labels[i] for i in top_idx[::-1]],
        mean_shap[top_idx[::-1]],
        color='#e74c3c', alpha=0.85
    )
    ax.set_xlabel('Mean |SHAP value| (pattern importance)')
    ax.set_title(f'SHAP — Top 15 Pattern Importances — {ds_name}')
    plt.tight_layout()
    save_fig(f'fig_shap_{ds_name}.png')
    print(f'    SHAP analysis complete. Top pattern: {pat_labels[top_idx[0]]}')

print('SHAP explainability function (Novelty 5) defined ✓')

SHAP explainability function (Novelty 5) defined ✓


## ⑪ Main Pipeline — Runs All Datasets

In [ ]:
def run_full_pipeline(name, df):
    print(f"\n{'═'*65}")
    print(f'  DATASET: {name}  |  Rows: {df.shape[0]}  Cols: {df.shape[1]}')
    print(f"{'═'*65}")

    # ── Pre-process ──────────────────────────────────────────
    df_clean, target = PREPROCESSORS[name](df)
    task             = TASK_TYPE[name]
    print(f'  Target: "{target}"  |  Task: {task}')
    X, y, feat_names = encode_features(df_clean, target)
    print(f'  Features: {len(feat_names)}  |  Samples: {len(y)}')
    print(f'  Class distribution: {dict(Counter(y))}')

    # ── Step 1: Frequent feature plot ────────────────────────
    print(f'\n  [Step 1] Frequent Feature Visualisation...')
    plot_frequent_features(df_clean, target, name)

    # ── Scale ────────────────────────────────────────────────
    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # ── CASE 1: All features, 10-fold CV ─────────────────────
    print(f'\n  [Step 2] CASE 1 — All Features (10-fold CV, 9 classifiers)')
    classifiers = build_classifiers(task)
    res_case1   = {}

    # Apply SMOTE for balanced training
    try:
        sm = SMOTE(random_state=42)
        X_sm, y_sm = sm.fit_resample(X_scaled, y)
        print(f'    SMOTE: {dict(Counter(y))} → {dict(Counter(y_sm))}')
    except Exception as e:
        print(f'    SMOTE skipped: {e}')
        X_sm, y_sm = X_scaled, y

    for cname, clf in classifiers.items():
        try:
            r = evaluate_classifier(clf, X_sm, y_sm, cv=10, task=task)
            res_case1[cname] = r
            print(f'    {cname:<5}| ACC={r["ACC"]:6.2f}% F1={r["F1"]:6.2f}% AUC={r["AUC"]:.3f}')
        except Exception as e:
            print(f'    {cname:<5}| ERROR: {e}')

    # ── NOVELTY 1: SPAM Pattern Mining ───────────────────────
    print(f'\n  [Step 3 — NOVELTY 1] SPAM Pattern Mining...')
    spam = SPAM(minsup=0.12, max_length=3)
    spam.fit(X_scaled, y, feature_names=feat_names)
    print(f'  Top patterns found:')
    print(spam.get_top_patterns_text(8))

    # ── NOVELTY 2: RuleGrowth ─────────────────────────────────
    print(f'\n  [Step 4 — NOVELTY 2] RuleGrowth Sequential Rules...')
    # Use SPAM's encoded sequences for rule mining
    MAX_ROWS = 2000
    X_s = X_scaled[:MAX_ROWS] if len(X_scaled) > MAX_ROWS else X_scaled
    seqs_for_rules = spam._encode(X_s, feat_names)
    rg = RuleGrowth(minsup=0.10, minconf=0.55)
    rg.fit(seqs_for_rules, n_total=len(seqs_for_rules), inv_token=spam.inv_token_)
    print(f'  Top rules:')
    rg.print_rules(5)

    # ── NOVELTY 3: Weighted Pattern Feature Matrix ────────────
    print(f'\n  [Step 5 — NOVELTY 3] Weighted Pattern Feature Matrix...')
    if len(spam.patterns_) > 0:
        X_weighted = build_weighted_pattern_matrix(X_scaled, spam, rg)
    else:
        print('    No patterns found — using scaled features for Case 2')
        X_weighted = X_scaled

    # ── CASE 2: Weighted pattern features, 10-fold CV ────────
    print(f'\n  [Step 6 — Novel] CASE 2 — Weighted Pattern Matrix (10-fold CV)')
    classifiers2 = build_classifiers(task)
    res_case2    = {}

    try:
        sm2 = SMOTE(random_state=42)
        X_wm_sm, y_wm_sm = sm2.fit_resample(X_weighted, y)
    except Exception:
        X_wm_sm, y_wm_sm = X_weighted, y

    X_wm_sm = np.clip(X_wm_sm, 0, None)

    for cname, clf in classifiers2.items():
        try:
            r = evaluate_classifier(clf, X_wm_sm, y_wm_sm, cv=10, task=task)
            res_case2[cname] = r
            print(f'    {cname:<5}| ACC={r["ACC"]:6.2f}% F1={r["F1"]:6.2f}% AUC={r["AUC"]:.3f}')
        except Exception as e:
            print(f'    {cname:<5}| ERROR: {e}')

    # ── NOVELTY 5: SHAP Explainability ────────────────────────
    print(f'\n  [Step 7 — NOVELTY 5] SHAP Explainability Layer...')
    try:
        run_shap_explainability(X_weighted, y, spam, name, task)
    except Exception as e:
        print(f'    SHAP error: {e}')

    # ── Plots ────────────────────────────────────────────────
    plot_case_comparison(res_case1, res_case2, name)
    plot_metrics_bar(res_case2, name, 'Case2-Weighted')

    return {
        'case1' : res_case1,
        'case2' : res_case2,
        'n_patterns': len(spam.patterns_),
        'n_rules'   : len(rg.rules_)
    }

print('Main pipeline function defined ✓')

Main pipeline function defined ✓


## ⑫ Run All Datasets

In [ ]:
ALL_RESULTS = {}

for name, df in DATASETS.items():
    try:
        result = run_full_pipeline(name, df)
        ALL_RESULTS[name] = result
    except Exception as e:
        print(f'\n  ERROR in {name}: {e}')
        import traceback
        traceback.print_exc()

print(f"\n{'='*65}")
print('  All datasets processed.')
print(f"  Figures saved: {len(SAVED_FIGS)}")
print(f"{'='*65}")


═════════════════════════════════════════════════════════════════
  DATASET: IBM-1  |  Rows: 1470  Cols: 35
═════════════════════════════════════════════════════════════════
  Target: "Attrition"  |  Task: binary
  Features: 30  |  Samples: 1470
  Class distribution: {np.int64(1): 237, np.int64(0): 1233}

  [Step 1] Frequent Feature Visualisation...
    Saved: fig2_IBM-1_frequent_features.png

  [Step 2] CASE 1 — All Features (10-fold CV, 9 classifiers)
    SMOTE: {np.int64(1): 237, np.int64(0): 1233} → {np.int64(1): 1233, np.int64(0): 1233}
    MNB  | ACC= 74.86% F1= 75.29% AUC=0.809
    GNB  | ACC= 69.62% F1= 73.06% AUC=0.801
    DT   | ACC= 84.87% F1= 85.15% AUC=0.849
    RF   | ACC= 94.49% F1= 94.34% AUC=0.986
    MLP  | ACC= 77.74% F1= 77.84% AUC=0.853
    SVM  | ACC= 88.97% F1= 89.21% AUC=0.957
    kNN  | ACC= 91.08% F1= 91.77% AUC=0.912
    LR   | ACC= 78.10% F1= 78.51% AUC=0.853
    XGB  | ACC= 93.19% F1= 92.97% AUC=0.975

  [Step 3 — NOVELTY 1] SPAM Pattern Mining...
    SPAM

## ⑬ Summary Heatmap & Final Results Table

In [ ]:
if len(ALL_RESULTS) >= 1:
    plot_summary_heatmap(ALL_RESULTS)

# ── Print comparison table ───────────────────────────────────
print('\n' + '='*75)
print('  FINAL RESULTS COMPARISON — Case 1 vs Case 2 (Best classifier per dataset)')
print('='*75)
print(f"{'Dataset':<8} {'Case':<8} {'Best Clf':<6} {'ACC':>7} {'P':>7} {'R':>7} {'F1':>7} {'AUC':>7}")
print('-'*75)

for ds, res in ALL_RESULTS.items():
    for case_key, case_label in [('case1', 'Case1'), ('case2', 'Case2')]:
        case_res = res.get(case_key, {})
        if not case_res:
            continue
        best_clf = max(case_res, key=lambda c: case_res[c].get('ACC', 0))
        r = case_res[best_clf]
        print(f"{ds:<8} {case_label:<8} {best_clf:<6} "
              f"{r['ACC']:>7.2f} {r['P']:>7.2f} {r['R']:>7.2f} "
              f"{r['F1']:>7.2f} {r['AUC']:>7.3f}")
    print('-'*75)

print('\nNovel additions used:')
print('  ✓ SPAM     — bitmap mining finds ALL frequent patterns (vs TKS top-k only)')
print('  ✓ RuleGrowth — incremental rule growth (vs ERMiner SCM approach)')
print('  ✓ Weighted Matrix — support × confidence weights (vs binary 0/1 matrix)')
print('  ✓ XGBoost  — 9th classifier added to the 8 base classifiers')
print('  ✓ SHAP     — pattern-level explainability layer (base paper had none)')

    Saved: fig_summary_heatmap.png

  FINAL RESULTS COMPARISON — Case 1 vs Case 2 (Best classifier per dataset)
Dataset  Case     Best Clf     ACC       P       R      F1     AUC
---------------------------------------------------------------------------
IBM-1    Case1    RF       94.49   96.86   91.97   94.34   0.986
IBM-1    Case2    SVM      95.29   95.05   95.62   95.32   0.990
---------------------------------------------------------------------------
HRA-4    Case1    RF       99.01   99.65   98.36   99.00   0.998
HRA-4    Case2    RF       96.93   96.26   97.66   96.95   0.988
---------------------------------------------------------------------------

Novel additions used:
  ✓ SPAM     — bitmap mining finds ALL frequent patterns (vs TKS top-k only)
  ✓ RuleGrowth — incremental rule growth (vs ERMiner SCM approach)
  ✓ Weighted Matrix — support × confidence weights (vs binary 0/1 matrix)
  ✓ XGBoost  — 9th classifier added to the 8 base classifiers
  ✓ SHAP     — pattern-level e

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'networkx', '-q'])
print('NetworkX installed ✓')

NetworkX installed ✓


### RuleGrowth Sequential Diagram Visualization
To visualize the sequential rules, we first need to re-create the `SPAM` and `RuleGrowth` objects for one of the datasets. We'll use the IBM-1 dataset, as it was successfully processed.

In [ ]:
import numpy as np

# Re-running processing for IBM-1 to get RuleGrowth object for visualization
print("Re-processing IBM-1 dataset for RuleGrowth visualization...")

name = 'IBM-1'
df = DATASETS[name]

# Pre-process
df_clean, target = PREPROCESSORS[name](df.copy())
X, y, feat_names = encode_features(df_clean, target)

# Scale
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Define SPEED_CONFIG (moved from a later cell or re-defined for this context)
SPEED_CONFIG = {
    'IBM-1' : {'max_smote': 3000, 'cv': 10, 'spam_rows': 3000},
    'HRD-2' : {'max_smote': 3000, 'cv': 10, 'spam_rows': 3000},
}

# SPAM Pattern Mining
spam = SPAM(minsup=0.12, max_length=3)
# Use the same sampling logic as in run_full_pipeline if needed
cfg = SPEED_CONFIG.get(name, {'max_smote': 3000, 'cv': 10, 'spam_rows': 2000})
X_spam = X_scaled[:cfg['spam_rows']] if len(X_scaled) > cfg['spam_rows'] else X_scaled
spam.fit(X_spam, y[:len(X_spam)], feature_names=feat_names)

# Filter patterns to >= 3 items (paper §4.3)
spam.patterns_ = [(p, s, pct) for p, s, pct in spam.patterns_ if len(p) >= 3]

# RuleGrowth Sequential Rules
seqs_for_rules = spam._encode(X_spam, feat_names)
rg = RuleGrowth(minsup=0.10, minconf=0.55) # Re-instantiate RuleGrowth with same parameters
rg.fit(seqs_for_rules, n_total=len(seqs_for_rules), inv_token=spam.inv_token_)

print("RuleGrowth object 'rg' created for IBM-1. Top rules:")
rg.print_rules(5)

Re-processing IBM-1 dataset for RuleGrowth visualization...
    SPAM → 2147 ALL-frequent patterns (minsup=0.12) in 1.25s ✓
    RuleGrowth → 20 rules (minsup=0.1, minconf=0.55) in 1.66s ✓
RuleGrowth object 'rg' created for IBM-1. Top rules:
    ['BusinessTravel=1', 'PercentSalaryHike=0'] → ['PerformanceRating=0'] | sup=10.14% | conf=1.0
    ['MaritalStatus=1'] → ['StockOptionLevel=0'] | sup=31.97% | conf=1.0
    ['MaritalStatus=1', 'BusinessTravel=1'] → ['StockOptionLevel=0'] | sup=22.24% | conf=1.0
    ['MaritalStatus=1', 'Department=1'] → ['StockOptionLevel=0'] | sup=10.41% | conf=1.0
    ['MaritalStatus=1', 'EducationField=0.2'] → ['StockOptionLevel=0'] | sup=13.67% | conf=1.0


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("Generating RuleGrowth sequential diagram visualization...")

# Create a directed graph
G = nx.DiGraph()

# Add nodes and edges from rules
for i, rule in enumerate(rg.rules_):
    antecedent_tokens = [rg.inv_.get(t, (0, 0, str(t)))[2] for t in rule['antecedent']]
    consequent_tokens = [rg.inv_.get(t, (0, 0, str(t)))[2] for t in rule['consequent']]

    # Create a unique node for each antecedent and consequent part of a rule
    ant_node = " & ".join(antecedent_tokens)
    con_node = " & ".join(consequent_tokens)

    # Add nodes (if they don't exist)
    G.add_node(ant_node)
    G.add_node(con_node)

    # Add edge with rule details as attributes
    G.add_edge(ant_node, con_node,
               support=rule['sup_pct'], confidence=rule['confidence'])

# Layout the graph
pos = nx.spring_layout(G, k=0.8, iterations=50, seed=42) # Adjust k for spacing

plt.figure(figsize=(18, 12)) # Increase figure size for better readability

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=4000, node_color='lightblue', alpha=0.9)

# Draw edges with labels
edge_labels = {
    (u, v): f"S:{d['support']}%, C:{d['confidence']:.2f}"
    for u, v, d in G.edges(data=True)
}
nx.draw_networkx_edges(G, pos, edge_color='gray', arrowsize=20, alpha=0.7)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=8)

# Draw node labels
node_labels = {node: node for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=9, font_weight='bold')

plt.title("RuleGrowth Sequential Rules Diagram (IBM-1 Dataset)", size=18)
plt.axis('off')
plt.tight_layout()
save_fig('fig_rulegrowth_diagram_IBM-1.png') # Save the figure
plt.show() # Display the figure

Generating RuleGrowth sequential diagram visualization...
    Saved: fig_rulegrowth_diagram_IBM-1.png


### RuleGrowth for HRA-4 Datasets

In [ ]:
if 'HRD-2' not in DATASETS:
    print(f"  HRD-2 not found in DATASETS. Attempting to load HRD-2 for visualization.")
    df_hrd2 = load_dataset('HRD-2')
    if df_hrd2 is not None:
        DATASETS['HRD-2'] = df_hrd2
    else:
        print(f"  Warning: Failed to load HRD-2. It will be skipped for RuleGrowth visualization.")

# Filter names to process based on what's actually loaded in DATASETS
names_to_process = [name for name in ['HRD-2', 'HRA-4'] if name in DATASETS]

for name in names_to_process:
    print(f"\nRe-processing {name} dataset for RuleGrowth visualization...")

    df = DATASETS[name]

    # Pre-process
    df_clean, target = PREPROCESSORS[name](df.copy())
    X, y, feat_names = encode_features(df_clean, target)

    # Scale
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # SPAM Pattern Mining
    spam = SPAM(minsup=0.12, max_length=3)
    cfg = SPEED_CONFIG.get(name, {'max_smote': 3000, 'cv': 10, 'spam_rows': 2000})
    X_spam = X_scaled[:cfg['spam_rows']] if len(X_scaled) > cfg['spam_rows'] else X_scaled
    spam.fit(X_spam, y[:len(X_spam)], feature_names=feat_names)

    # Filter patterns to >= 3 items (paper §4.3)
    spam.patterns_ = [(p, s, pct) for p, s, pct in spam.patterns_ if len(p) >= 3]
    print(f"  Patterns after length filter (>=3): {len(spam.patterns_)}")

    # RuleGrowth Sequential Rules
    seqs_for_rules = spam._encode(X_spam, feat_names)
    rg = RuleGrowth(minsup=0.10, minconf=0.55)
    rg.fit(seqs_for_rules, n_total=len(seqs_for_rules), inv_token=spam.inv_token_)

    print(f"RuleGrowth object 'rg' created for {name}. Top rules:")
    rg.print_rules(5)

  HRD-2 not found in DATASETS. Attempting to load HRD-2 for visualization.
  ✗ HRD-2 NOT FOUND — skipping.

Re-processing HRA-4 dataset for RuleGrowth visualization...
    SPAM → 402 ALL-frequent patterns (minsup=0.12) in 0.18s ✓
  Patterns after length filter (>=3): 246
    RuleGrowth → 20 rules (minsup=0.1, minconf=0.55) in 3.57s ✓
RuleGrowth object 'rg' created for HRA-4. Top rules:
    ['satisfaction_level=0.32', 'last_evaluation=0.27'] → ['promotion_last_5years=0'] | sup=10.85% | conf=1.0
    ['satisfaction_level=0.32', 'number_project=0'] → ['promotion_last_5years=0'] | sup=20.8% | conf=1.0
    ['satisfaction_level=0.32', 'average_montly_hours=0.29'] → ['promotion_last_5years=0'] | sup=10.55% | conf=1.0
    ['satisfaction_level=0.32', 'time_spend_company=0.12'] → ['promotion_last_5years=0'] | sup=21.1% | conf=1.0
    ['satisfaction_level=0.32', 'last_evaluation=0.22'] → ['promotion_last_5years=0'] | sup=12.2% | conf=1.0


### RuleGrowth Sequential Diagram Visualization for HRA-4

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("Generating RuleGrowth sequential diagram visualization for HRA-4...")

# Create a directed graph
G_hra4 = nx.DiGraph()

# Add nodes and edges from rules
# The 'rg' object here should be from the last iteration of cell 1f31b200, which is HRA-4
for i, rule in enumerate(rg.rules_):
    antecedent_tokens = [rg.inv_.get(t, (0, 0, str(t)))[2] for t in rule['antecedent']]
    consequent_tokens = [rg.inv_.get(t, (0, 0, str(t)))[2] for t in rule['consequent']]

    # Create a unique node for each antecedent and consequent part of a rule
    ant_node = " & ".join(antecedent_tokens)
    con_node = " & ".join(consequent_tokens)

    # Add nodes (if they don't exist)
    G_hra4.add_node(ant_node)
    G_hra4.add_node(con_node)

    # Add edge with rule details as attributes
    G_hra4.add_edge(ant_node, con_node,
               support=rule['sup_pct'], confidence=rule['confidence'])

# Layout the graph
pos_hra4 = nx.spring_layout(G_hra4, k=0.8, iterations=50, seed=42) # Adjust k for spacing

plt.figure(figsize=(20, 14)) # Increase figure size for better readability

# Draw nodes
nx.draw_networkx_nodes(G_hra4, pos_hra4, node_size=4000, node_color='lightcoral', alpha=0.9)

# Draw edges with labels
edge_labels_hra4 = {
    (u, v): f"S:{d['support']}%, C:{d['confidence']:.2f}"
    for u, v, d in G_hra4.edges(data=True)
}
nx.draw_networkx_edges(G_hra4, pos_hra4, edge_color='gray', arrowsize=20, alpha=0.7)
nx.draw_networkx_edge_labels(G_hra4, pos_hra4, edge_labels=edge_labels_hra4, font_color='darkgreen', font_size=8)

# Draw node labels
node_labels_hra4 = {node: node for node in G_hra4.nodes()}
nx.draw_networkx_labels(G_hra4, pos_hra4, labels=node_labels_hra4, font_size=9, font_weight='bold')

plt.title("RuleGrowth Sequential Rules Diagram (HRA-4 Dataset)", size=18)
plt.axis('off')
plt.tight_layout()
save_fig('fig_rulegrowth_diagram_HRA-4.png') # Save the figure
plt.show() # Display the figure

Generating RuleGrowth sequential diagram visualization for HRA-4...
    Saved: fig_rulegrowth_diagram_HRA-4.png


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. COMPARISON TABLE
# -----------------------------
data = {
    "Aspect": ["Pattern Mining", "Rule Mining", "Features", "Model", "Explainability"],
    "Existing": ["TKS", "ERMiner", "Binary (0/1)", "DT/RF/SVM", "No"],
    "Proposed": ["SPAM", "RuleGrowth", "Weighted", "XGBoost", "SHAP"]
}

df_table = pd.DataFrame(data)

print("\n===== COMPARISON TABLE =====\n")
display(df_table)

# -----------------------------
# 2. HEATMAP (IMPROVEMENT VIEW)
# -----------------------------
heatmap_data = {
    "Pattern Mining": [0, 1],
    "Rule Mining": [0, 1],
    "Features": [0, 1],
    "Model": [0, 1],
    "Explainability": [0, 1]
}

df_heatmap = pd.DataFrame(heatmap_data, index=["Existing", "Proposed"])

plt.figure()
sns.heatmap(df_heatmap, annot=True, fmt="d")
plt.title("Improvement Heatmap (0=Basic, 1=Improved)")
plt.show()

# -----------------------------
# 3. BAR CHART (SYSTEM COMPARISON)
# -----------------------------
labels = ["Patterns", "Rules", "Features", "Model", "Explainability"]

existing = [1, 1, 1, 1, 0]
proposed = [2, 2, 2, 2, 2]

x = np.arange(len(labels))

plt.figure()
plt.bar(x - 0.2, existing, width=0.4, label="Existing")
plt.bar(x + 0.2, proposed, width=0.4, label="Proposed")

plt.xticks(x, labels)
plt.title("Existing vs Proposed System")
plt.legend()
plt.show()

# -----------------------------
# 4. ACCURACY COMPARISON (EXAMPLE)
# -----------------------------
models = ["DT", "RF", "SVM", "XGBoost"]

existing_acc = [85, 94, 88, 93]
proposed_acc = [85.4, 93.2, 95.2, 92.3]

x = np.arange(len(models))

plt.figure()
plt.bar(x - 0.2, existing_acc, width=0.4, label="Existing")
plt.bar(x + 0.2, proposed_acc, width=0.4, label="Proposed")

plt.xticks(x, models)
plt.title("Model Accuracy Comparison")
plt.legend()
plt.show()


===== COMPARISON TABLE =====



,Aspect,Existing,Proposed
0,Pattern Mining,TKS,SPAM
1,Rule Mining,ERMiner,RuleGrowth
2,Features,Binary (0/1),Weighted
3,Model,DT/RF/SVM,XGBoost
4,Explainability,No,SHAP


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# -----------------------------
# 1. COMBINE RESULTS INTO TABLE
# -----------------------------
rows = []

for dataset, res in ALL_RESULTS.items():
    for model in res['case1']:
        if model in res['case2']:
            rows.append({
                "Dataset": dataset,
                "Model": model,
                "Case1_ACC": res['case1'][model]['ACC'],
                "Case2_ACC": res['case2'][model]['ACC']
            })

df = pd.DataFrame(rows)

print("\n===== COMPARISON TABLE =====\n")
display(df.head())

# -----------------------------
# 2. DATASET-WISE ACCURACY COMPARISON
# -----------------------------
for dataset in df["Dataset"].unique():
    temp = df[df["Dataset"] == dataset]

    x = np.arange(len(temp))

    plt.figure()
    plt.bar(x - 0.2, temp["Case1_ACC"], width=0.4, label="Case 1")
    plt.bar(x + 0.2, temp["Case2_ACC"], width=0.4, label="Case 2")

    plt.xticks(x, temp["Model"], rotation=45)
    plt.title(f"{dataset} - Model Accuracy Comparison")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------------
# 3. OVERALL AVERAGE COMPARISON
# -----------------------------
avg_case1 = df["Case1_ACC"].mean()
avg_case2 = df["Case2_ACC"].mean()

plt.figure()
plt.bar(["Case 1", "Case 2"], [avg_case1, avg_case2])
plt.title("Overall Accuracy Comparison")
plt.ylabel("Average Accuracy (%)")
plt.show()

# -----------------------------
# 4. PATTERN & RULE COUNT
# -----------------------------
datasets = []
patterns = []
rules = []

for dataset, res in ALL_RESULTS.items():
    datasets.append(dataset)
    patterns.append(res['n_patterns'])
    rules.append(res['n_rules'])

x = np.arange(len(datasets))

plt.figure()
plt.bar(x - 0.2, patterns, width=0.4, label="Patterns")
plt.bar(x + 0.2, rules, width=0.4, label="Rules")

plt.xticks(x, datasets)
plt.title("Patterns vs Rules per Dataset")
plt.legend()
plt.show()


===== COMPARISON TABLE =====



,Dataset,Model,Case1_ACC,Case2_ACC
0,IBM-1,MNB,74.86,76.56
1,IBM-1,GNB,69.62,81.26
2,IBM-1,DT,84.87,85.81
3,IBM-1,RF,94.49,93.27
4,IBM-1,MLP,77.74,92.78


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pickle, matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/project_outputs'
os.makedirs(save_path, exist_ok=True)

# Save ALL_RESULTS
with open(f'{save_path}/ALL_RESULTS.pkl', 'wb') as f:
    pickle.dump(ALL_RESULTS, f)

# Save current figure (if any)
plt.savefig(f'{save_path}/last_plot.png')

print("All outputs saved to Google Drive:", save_path)

In [ ]:
import shutil, os
from google.colab import files

folder = "final_outputs"
os.makedirs(folder, exist_ok=True)

# Move only png and csv files
for f in os.listdir():
    if f.endswith(".png") or f.endswith(".csv"):
        shutil.copy(f, folder)

# Zip and download
shutil.make_archive("final_outputs", 'zip', folder)
files.download("final_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_csv('/content/drive/MyDrive/results.csv', index=False)

In [ ]:
# Re-running the pipeline for HRA-4 to generate updated SHAP visualization
print("\n" + "="*65)
print('  Attempting to re-run pipeline for HRA-4 to update SHAP visualization')
print("="*65)

name_to_rerun = 'HRA-4'

if name_to_rerun in DATASETS:
    print(f"  Dataset '{name_to_rerun}' found in DATASETS. Initiating pipeline...")
    try:
        # The DATASETS dictionary already contains the loaded dataframes
        result = run_full_pipeline(name_to_rerun, DATASETS[name_to_rerun])
        ALL_RESULTS[name_to_rerun] = result
        print(f"\nSuccessfully re-ran pipeline for {name_to_rerun}. SHAP plot generated and saved as fig_shap_HRA-4.png")
    except Exception as e:
        print(f'\n  ERROR during pipeline re-run for {name_to_rerun}: {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()
        print(f"\n  Pipeline re-run for {name_to_rerun} failed. Please check the traceback above.")
else:
    print(f"Dataset {name_to_rerun} not found in DATASETS. Please ensure all datasets are loaded before re-running this cell.")


In [ ]:
import pandas as pd

# Load the raw HRA-4 dataset to inspect the 'left' column directly
df_hra4_raw = pd.read_csv('/content/hr_data.csv')

print("\nValue counts for the original 'left' column in HRA-4 dataset (hr_data.csv):")
print(df_hra4_raw['left'].value_counts())

# Also check the unique values and data type
print("\nUnique values in 'left' column:", df_hra4_raw['left'].unique())
print("Data type of 'left' column:", df_hra4_raw['left'].dtype)


In [ ]:
print(f"\n{'='*50}")
print(f"Results for HRA-4 Dataset after fix")
print(f"{'='*50}")

hra4_results = ALL_RESULTS.get('HRA-4')

if hra4_results:
    print(f"Number of patterns found: {hra4_results.get('n_patterns')}")
    print(f"Number of rules found: {hra4_results.get('n_rules')}")

    print("\nCase 1 (All Features) Results:")
    for clf_name, metrics in hra4_results.get('case1', {}).items():
        print(f"  {clf_name:<5}| ACC={metrics['ACC']:6.2f}% F1={metrics['F1']:6.2f}% AUC={metrics['AUC']:.3f}")

    print("\nCase 2 (Weighted Pattern Matrix) Results:")
    for clf_name, metrics in hra4_results.get('case2', {}).items():
        print(f"  {clf_name:<5}| ACC={metrics['ACC']:6.2f}% F1={metrics['F1']:6.2f}% AUC={metrics['AUC']:.3f}")
else:
    print("HRA-4 results not found in ALL_RESULTS. Please ensure the pipeline for HRA-4 was re-run.")